# Principal component analysis (PCA)

_Machine Learning_

**Find the directions of most spread. Keep them, drop the rest.**

Data with many features is hard to see and slow to process.

     PCA finds the few directions where the data spreads out the most.

---

This notebook is a practice scaffold for the **Principal component analysis (PCA)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — scikit-learn

### Step 1 — Make a dataset and fit PCA on all components

We generate 400 synthetic examples with 8 features (3 genuinely informative). Fitting PCA with all 8 components rotates the data onto new axes ordered by how much variance each captures, without dropping anything yet — that lets us inspect the full variance breakdown.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.decomposition import PCA

X, _ = make_classification(n_samples=400, n_features=8, n_informative=3,
                           random_state=0)

pca = PCA(n_components=8, random_state=0).fit(X)

### Step 2 — Inspect variance explained per component

`explained_variance_ratio_` tells us what fraction of the total spread each new axis captures, in descending order. The cumulative sum shows how much of the data we'd retain by keeping the first few components — the key number for deciding how far to reduce.

In [ ]:
ratio = pca.explained_variance_ratio_
print("variance explained per component:", np.round(ratio, 3))

cumulative = np.cumsum(ratio)
print("cumulative:", np.round(cumulative, 3))

### Step 3 — Project down to 2 dimensions

Now we actually reduce: fitting PCA with `n_components=2` keeps only the two highest-variance directions and transforms every example into that 2-D space. We confirm the new shape and report how much of the original variance those two components preserve.

In [ ]:
# project down to 2 dimensions
X2 = PCA(n_components=2, random_state=0).fit_transform(X)
print("reduced shape:", X2.shape)

kept_variance = round(float(cumulative[1]), 3)
print("kept variance with 2 comps:", kept_variance)

## Visualize the data & results

_Handwritten digits live in 64 pixels. What 2 directions capture the most spread, and how much variance do they keep?_

### Step 1 — Standardise the digit pixels and fit PCA

We load 1797 real 8x8 handwritten digit images (64 pixels each) and standardise every pixel so no single pixel dominates by scale. Fitting PCA with 10 components finds the 10 directions of greatest pixel-intensity spread across all the images.

In [ ]:
# 1797 real 8x8 handwritten digit images, 64 pixels each
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

digits = load_digits()
X = StandardScaler().fit_transform(digits.data)
y = digits.target
pca = PCA(n_components=10, random_state=0).fit(X)

### Step 2 — Project the images onto their first two components

We transform the 64-pixel images into the new PCA space and keep only the first two components. Those two numbers per image are the directions of most spread — enough to plot 64-dimensional images as ordinary points on a 2-D scatter.

In [ ]:
P = pca.transform(X)[:, :2]

### Step 3 — Plot the 2-D projection and the variance bars

The left panel scatters digits 0 and 1 in the two-component space; even with only 2 of 64 dimensions they separate clearly. The right panel shows how much variance each of the first eight components captures — the first few bars dwarf the rest, which is exactly why PCA works.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Left — two digit classes in 2-D PCA space
for label, color, name in [(0, "#4ea1ff", "digit 0"), (1, "#7ee787", "digit 1")]:
    pts = P[y == label]
    ax1.scatter(pts[:, 0], pts[:, 1], c=color, label=name, edgecolor="k")
ax1.set_xlabel("PC 1")
ax1.set_ylabel("PC 2")
ax1.set_title("Digits projected to 2-D")
ax1.legend()

# Right — variance explained by the first eight components
ratio = pca.explained_variance_ratio_[:8]
component_labels = ["PC%d" % (i + 1) for i in range(8)]
ax2.bar(component_labels, ratio, color="#ffb454")
ax2.set_title("Variance explained per component")

plt.show()